In [1]:
import pandas as pd
import json
import torch
import re
from collections import Counter

train_path = "/teamspace/studios/this_studio/T2I_Captions/train_captions.jsonl"
test_path = "/teamspace/studios/this_studio/T2I_Captions/test_captions.jsonl"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Captions Preparation

In [2]:
train_rows = []
with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        train_rows.append(json.loads(line))

In [3]:
df = pd.DataFrame(train_rows)
print(df.shape)
df.head()

(11647, 2)


,file_name,raw_caption
0,00772_00.jpg,"T-shirt, white, cotton, short sleeves, crew ne..."
1,09687_00.jpg,"tank top, light pink, jersey knit, sleeveless,..."
2,05609_00.jpg,"Garment category: T-shirt, Colors: Navy blue, ..."
3,07971_00.jpg,"Garment category: Turtleneck sweater, Colors: ..."
4,13463_00.jpg,"T-shirt,pink,stretch cotton,short sleeves,V-ne..."


In [4]:
test_rows = []
with open(test_path, "r", encoding="utf-8") as f:
    for line in f:
        test_rows.append(json.loads(line))

In [5]:
test_df = pd.DataFrame(test_rows)
print(test_df.shape)
test_df.head()

(2032, 2)


,file_name,raw_caption
0,00888_00.jpg,"white long-sleeved top, white, smooth, long sl..."
1,08428_00.jpg,"blouse, peach, sheer, long sleeves, high neck,..."
2,02653_00.jpg,"tank top, white, printed, spaghetti straps, sc..."
3,11206_00.jpg,"tank top, black, jersey, short, scoop neck, lo..."
4,00725_00.jpg,"Garment category: T-shirt, Colors: light beige..."


In [6]:
df.isna().sum()

file_name      0
raw_caption    0
dtype: int64

In [7]:
df["raw_caption"].sample(15)

288      tank top, black, sheer, sleeveless, v-neck, lo...
5081     Garment category: Shift dress, Colors: Black, ...
10616    tank top, navy blue, white, red, striped, slee...
4237     Garment category: T-shirt, Colors: Black, Whit...
4094     T-shirt, red, cotton, short sleeves, crew neck...
3162     Garment category: Top, Colors: Black, Material...
2938     garment category: swimsuit, colors: black, mat...
11446    long-sleeved t-shirt, light blue and white str...
5490     Garment category: Tunic, Colors: Navy blue, Ma...
4435     bodysuit, burgundy, ribbed knit, long sleeves,...
4205     football jersey, light green, mesh-like, short...
2603     Garment category: Tunic, Colors: Black, Materi...
801      Garment category: T-shirt, Colors: white, yell...
8894     Garment category: Top, Colors: Maroon, Materia...
7511     long-sleeved top, black and white, knit, long ...
Name: raw_caption, dtype: object

## Data Splitting

In [8]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.10, random_state=42, shuffle=True)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 10482
Validation: 1165
Test: 2032


In [9]:
train_df.head()

,file_name,raw_caption
0,08945_00.jpg,"garment category: camisole, colors: black, mat..."
1,06064_00.jpg,"blouse, red, crepe-like, short, scoop neck, lo..."
2,02638_00.jpg,"T-shirt, gray, cotton, short sleeves, scoop ne..."
3,01335_00.jpg,"blouse,yellow,likely silk,full sleeves,v-neckl..."
4,03561_00.jpg,"crop top, cream, black, orange, leopard print,..."


In [10]:
val_df.head()

,file_name,raw_caption
0,11255_00.jpg,"blouse, black, lace, long, high neck, fitted, ..."
1,03456_00.jpg,"camisole, black, lace, spaghetti straps, scoop..."
2,11068_00.jpg,"Garment category: T-shirt, Colors: Black, Mate..."
3,12833_00.jpg,"tank top, black, smooth jersey, sleeveless, sc..."
4,06764_00.jpg,"T-shirt, white, cotton, short sleeves, crew ne..."


## Unify Captions Format

In [11]:
def normalize_caption_format(caption):

    caption = caption.strip()

    # field: value  --> value
    caption = re.sub(
        r'(garment category|colors?|material|sleeve length|neckline or collar|fit|fabric pattern|graphic prints|logos?|embroidery|text on the garment|decorations?|closures?|fashion style)\s*:\s*',
        '',
        caption,
        flags=re.IGNORECASE
    )

    # field=value --> value
    caption = re.sub(
        r'(garment category|colors?|material|sleeve length|neckline or collar|fit|fabric pattern|graphic prints|logos?|embroidery|text on the garment|decorations?|closures?|fashion style)\s*=\s*',
        '',
        caption,
        flags=re.IGNORECASE
    )

    return caption

In [12]:
train_df["caption_v1"] = train_df["raw_caption"].apply(normalize_caption_format)
val_df["caption_v1"] = val_df["raw_caption"].apply(normalize_caption_format)
test_df["caption_v1"] = test_df["raw_caption"].apply(normalize_caption_format)

In [16]:
train_df[["raw_caption", "caption_v1"]].sample(5)

,raw_caption,caption_v1
1633,"long-sleeved t-shirt, light blue and white str...","long-sleeved t-shirt, light blue and white str..."
94,"white long-sleeved top, white, smooth, long sl...","white long-sleeved top, white, smooth, long sl..."
1881,"maternity top, gray, jersey knit, long sleeves...","maternity top, gray, jersey knit, long sleeves..."
6996,"Garment category: blouse, Colors: yellow, Mate...","blouse, yellow, smooth, short, V-neck, loose, ..."
5288,"blouse, dark blue, lightweight, long sleeves, ...","blouse, dark blue, lightweight, long sleeves, ..."


In [14]:
val_df[["raw_caption", "caption_v1"]].sample(5)

,raw_caption,caption_v1
92,"maternity top, navy blue, stretchy, long sleev...","maternity top, navy blue, stretchy, long sleev..."
48,"T-shirt, pink, white, and gray, short sleeves,...","T-shirt, pink, white, and gray, short sleeves,..."
1082,"tank top, light pink, jersey, sleeveless, scoo...","tank top, light pink, jersey, sleeveless, scoo..."
1008,"sweater, blue, white, textured knit, long slee...","sweater, blue, white, textured knit, long slee..."
51,"Garment category: Top, Colors: Green, Material...","Top, Green, Ribbed knit, Short, Square, Fitted..."


In [15]:
test_df[["raw_caption", "caption_v1"]].sample(5)

,raw_caption,caption_v1
860,"Garment category: T-shirt, Colors: Red, Materi...","T-shirt, Red, Not specified, Cap sleeves, Roun..."
1736,"white t-shirt, white, short sleeves, v-neck, l...","white t-shirt, white, short sleeves, v-neck, l..."
45,"white turtleneck long-sleeved top, white, smoo...","white turtleneck long-sleeved top, white, smoo..."
1873,"Garment category: T-shirt, Colors: Black, Whit...","T-shirt, Black, White, Cotton, Short, Crew nec..."
1279,"Garment category: dress, Colors: black, red, p...","dress, black, red, pink, white, orange, green,..."


## Remove Negative Attributes

In [17]:
counter = Counter()

for caption in train_df["caption_v1"]:
    attrs = [x.strip().lower() for x in caption.split(",")]

    counter.update(attrs)

unique_attrs = pd.DataFrame(
    counter.items(),
    columns=["attribute", "count"]
)

unique_attrs = unique_attrs.sort_values(
    "count",
    ascending=False
)

unique_attrs

,attribute,count
7,none,14719
32,no closures,5485
29,no embroidery,5391
28,no logos,4820
6,solid,4655
...,...,...
3627,black logo with white text,1
3626,abstract wave design,1
3625,abstract wave print,1
3623,hands and wi-fi symbol graphic print,1


In [18]:
unique_attrs[
    unique_attrs["attribute"]
    .str.lower()
    .str.contains("not specified", na=False)
].sort_values("count", ascending=False)

,attribute,count
82,not specified,1286
1771,not specified but appears to be cotton,6
1869,not specified but appears to be a smooth,4
2697,not specified but appears to be a blend of lac...,2
5697,not specified but appears to be a smooth knit,1
5702,not specified but appears to be a solid color,1
5539,not specified but appears to be a lightweight ...,1
6397,not specified but appears to be a knit fabric,1
2490,material not specified,1
703,not specified but appears to be cotton or line...,1


In [19]:
unique_attrs[unique_attrs["attribute"].str.strip().eq("none")]

,attribute,count
7,none,14719


In [20]:
unique_attrs[
    unique_attrs["attribute"].str.lower().str.startswith("no ")
].sort_values("count", ascending=False)

,attribute,count
32,no closures,5485
29,no embroidery,5391
28,no logos,4820
30,no text,3062
27,no prints,2723
...,...,...
2235,no visible,1
2153,"no text other than ""elcy""",1
2352,no text other than brand name and size,1
1889,"no text other than ""rykiel""",1


In [21]:
NEGATIVE_EXACT = {
    "none",
    "not specified",

    "no closures",
    "no closure",

    "no embroidery",

    "no logos",
    "no logo",

    "no text",
    "no text on the garment",

    "no prints",
    "no print",

    "no graphic prints",
    "no graphic print",

    "no pattern",
    "no patterns",

    "no decoration",
    "no decorations",
}


def remove_negative_attributes(caption):
    attrs = [x.strip() for x in caption.split(",")]

    cleaned = []

    for attr in attrs:

        attr_norm = attr.strip().lower()

        if attr_norm in NEGATIVE_EXACT:
            continue

        if attr_norm == "":
            continue

        cleaned.append(attr.strip())

    return ", ".join(cleaned)


In [22]:
train_df["caption_v2"] = train_df["caption_v1"].apply(remove_negative_attributes)
val_df["caption_v2"] = val_df["caption_v1"].apply(remove_negative_attributes)
test_df["caption_v2"] = test_df["caption_v1"].apply(remove_negative_attributes)

In [23]:
train_df[["caption_v1", "caption_v2"]].sample(5)

,caption_v1,caption_v2
6506,"long-sleeved top, navy blue with white polka d...","long-sleeved top, navy blue with white polka d..."
985,"sweater, black, knit, long sleeves, v-neck, fi...","sweater, black, knit, long sleeves, v-neck, fi..."
5360,"white long-sleeved top, white, solid, long sle...","white long-sleeved top, white, solid, long sle..."
9797,"blouse, olive green, black, purple, beige, lon...","blouse, olive green, black, purple, beige, lon..."
5255,"swimsuit, black, ribbed, none, strapless, high...","swimsuit, black, ribbed, strapless, high-waist..."


In [24]:
val_df[["caption_v1", "caption_v2"]].sample(5)

,caption_v1,caption_v2
174,"T-shirt, Coral, Not specified, 3/4, Scoop neck...","T-shirt, Coral, 3/4, Scoop neck, Slim, Solid, ..."
583,"T-shirt, light gray heather, cotton, short sle...","T-shirt, light gray heather, cotton, short sle..."
488,"blouse, white, silk-like, long sleeves, V-neck...","blouse, white, silk-like, long sleeves, V-neck..."
858,"white t-shirt, white, cotton, short sleeves, c...","white t-shirt, white, cotton, short sleeves, c..."
117,"Blouse, Multicolored (pink, yellow, black, whi...","Blouse, Multicolored (pink, yellow, black, whi..."


In [26]:
test_df[["caption_v1", "caption_v2"]].sample(5)

,caption_v1,caption_v2
1886,"bodysuit, dark gray, stretchy knit, long sleev...","bodysuit, dark gray, stretchy knit, long sleev..."
37,"T-shirt, White, Cotton, Short, Crew neck, Rela...","T-shirt, White, Cotton, Short, Crew neck, Rela..."
1704,"blouse, red, satin-like, 3/4 sleeves, deep V-n...","blouse, red, satin-like, 3/4 sleeves, deep V-n..."
524,"sleeveless top, white, blue, not specified, sl...","sleeveless top, white, blue, sleeveless, round..."
471,"long-sleeved top, teal, knit, long sleeves, hi...","long-sleeved top, teal, knit, long sleeves, hi..."


## Sleeve Mapping

In [27]:
counter = Counter()

for caption in train_df["caption_v2"]:
    attrs = [x.strip().lower() for x in caption.split(",")]

    counter.update(attrs)

unique_attrs_v2 = pd.DataFrame(
    counter.items(),
    columns=["attribute", "count"]
)

unique_attrs_v2 = unique_attrs_v2.sort_values(
    "count",
    ascending=False
)

unique_attrs_v2

,attribute,count
6,solid,4655
38,fitted,4572
1,black,3270
18,t-shirt,3111
55,white,2854
...,...,...
3613,black logo with white text,1
3612,abstract wave design,1
3611,abstract wave print,1
3609,hands and wi-fi symbol graphic print,1


In [28]:
patterns = [
    "short",
    "long",
    "3/4",
    "sleeve",
    "sleeveless",
    "strapless",
    "spaghetti",
    "cap"
]

sleeve_attrs = unique_attrs_v2[
    unique_attrs_v2["attribute"].str.contains(
        "|".join(patterns),
        case=False,
        na=False
    )
].sort_values(
    "count",
    ascending=False
)

sleeve_attrs

,attribute,count
51,long sleeves,2379
12,short,2240
42,sleeveless,1698
21,short sleeves,1670
76,long-sleeved top,701
...,...,...
6194,black long-sleeved ribbed sweater,1
6352,red cursive text on sleeves,1
6282,"""calvin"" text on sleeves",1
6024,yellow and black decorative patterns on sleeves,1


In [29]:
sleeve_attrs.head(30)

,attribute,count
51,long sleeves,2379
12,short,2240
42,sleeveless,1698
21,short sleeves,1670
76,long-sleeved top,701
66,long,500
106,3/4,421
214,long-sleeved t-shirt,372
156,3/4 sleeves,286
88,sleeveless top,233


In [30]:
unique_attrs_v2[
    unique_attrs_v2["attribute"].isin(
        [
            "short",
            "short sleeve",
            "short sleeves",
            "short-sleeved",

            "long",
            "long sleeve",
            "long sleeves",
            "long-sleeved",

            "3/4",
            "3/4 sleeve",
            "3/4 sleeves",

            "sleeveless",
            "no sleeves",

            "full sleeves",
            "cap sleeves",
            "spaghetti straps",
            "strapless",
        ]
    )
].sort_values("count", ascending=False)

,attribute,count
51,long sleeves,2379
12,short,2240
42,sleeveless,1698
21,short sleeves,1670
66,long,500
106,3/4,421
156,3/4 sleeves,286
57,3/4 sleeve,215
62,spaghetti straps,187
228,short-sleeved,174


In [31]:
SLEEVE_MAP = {
    # short sleeves
    "short": "short sleeves",
    "short sleeve": "short sleeves",
    "short sleeves": "short sleeves",
    "short-sleeved": "short sleeves",

    # long sleeves
    "long": "long sleeves",
    "long sleeves": "long sleeves",
    "long-sleeved": "long sleeves",
    "full sleeves": "long sleeves",

    # 3/4 sleeves
    "3/4": "3/4 sleeves",
    "3/4 sleeve": "3/4 sleeves",
    "3/4 sleeves": "3/4 sleeves",

    # sleeveless
    "no sleeves": "sleeveless",
    "sleeveless": "sleeveless",
}

In [32]:
def normalize_sleeves(caption):
    attrs = [x.strip() for x in caption.split(",")]

    normalized = []

    for attr in attrs:
        attr_norm = attr.strip().lower()

        if attr_norm in SLEEVE_MAP:
            normalized.append(SLEEVE_MAP[attr_norm])
        else:
            normalized.append(attr.strip())

    return ", ".join(normalized)

In [33]:
train_df["caption_v3"] = train_df["caption_v2"].apply(normalize_sleeves)
val_df["caption_v3"] = val_df["caption_v2"].apply(normalize_sleeves)
test_df["caption_v3"] = test_df["caption_v2"].apply(normalize_sleeves)

In [34]:
train_df.head()

,file_name,raw_caption,caption_v1,caption_v2,caption_v3
0,08945_00.jpg,"garment category: camisole, colors: black, mat...","camisole, black, lace and possibly silk, strap...","camisole, black, lace and possibly silk, strap...","camisole, black, lace and possibly silk, strap..."
1,06064_00.jpg,"blouse, red, crepe-like, short, scoop neck, lo...","blouse, red, crepe-like, short, scoop neck, lo...","blouse, red, crepe-like, short, scoop neck, lo...","blouse, red, crepe-like, short sleeves, scoop ..."
2,02638_00.jpg,"T-shirt, gray, cotton, short sleeves, scoop ne...","T-shirt, gray, cotton, short sleeves, scoop ne...","T-shirt, gray, cotton, short sleeves, scoop ne...","T-shirt, gray, cotton, short sleeves, scoop ne..."
3,01335_00.jpg,"blouse,yellow,likely silk,full sleeves,v-neckl...","blouse,yellow,likely silk,full sleeves,v-neckl...","blouse, yellow, likely silk, full sleeves, v-n...","blouse, yellow, likely silk, long sleeves, v-n..."
4,03561_00.jpg,"crop top, cream, black, orange, leopard print,...","crop top, cream, black, orange, leopard print,...","crop top, cream, black, orange, leopard print,...","crop top, cream, black, orange, leopard print,..."


In [35]:
val_df.head()

,file_name,raw_caption,caption_v1,caption_v2,caption_v3
0,11255_00.jpg,"blouse, black, lace, long, high neck, fitted, ...","blouse, black, lace, long, high neck, fitted, ...","blouse, black, lace, long, high neck, fitted, ...","blouse, black, lace, long sleeves, high neck, ..."
1,03456_00.jpg,"camisole, black, lace, spaghetti straps, scoop...","camisole, black, lace, spaghetti straps, scoop...","camisole, black, lace, spaghetti straps, scoop...","camisole, black, lace, spaghetti straps, scoop..."
2,11068_00.jpg,"Garment category: T-shirt, Colors: Black, Mate...","T-shirt, Black, Sheer, Short, Round, Loose, St...","T-shirt, Black, Sheer, Short, Round, Loose, St...","T-shirt, Black, Sheer, short sleeves, Round, L..."
3,12833_00.jpg,"tank top, black, smooth jersey, sleeveless, sc...","tank top, black, smooth jersey, sleeveless, sc...","tank top, black, smooth jersey, sleeveless, sc...","tank top, black, smooth jersey, sleeveless, sc..."
4,06764_00.jpg,"T-shirt, white, cotton, short sleeves, crew ne...","T-shirt, white, cotton, short sleeves, crew ne...","T-shirt, white, cotton, short sleeves, crew ne...","T-shirt, white, cotton, short sleeves, crew ne..."


In [36]:
test_df.head()

,file_name,raw_caption,caption_v1,caption_v2,caption_v3
0,00888_00.jpg,"white long-sleeved top, white, smooth, long sl...","white long-sleeved top, white, smooth, long sl...","white long-sleeved top, white, smooth, long sl...","white long-sleeved top, white, smooth, long sl..."
1,08428_00.jpg,"blouse, peach, sheer, long sleeves, high neck,...","blouse, peach, sheer, long sleeves, high neck,...","blouse, peach, sheer, long sleeves, high neck,...","blouse, peach, sheer, long sleeves, high neck,..."
2,02653_00.jpg,"tank top, white, printed, spaghetti straps, sc...","tank top, white, printed, spaghetti straps, sc...","tank top, white, printed, spaghetti straps, sc...","tank top, white, printed, spaghetti straps, sc..."
3,11206_00.jpg,"tank top, black, jersey, short, scoop neck, lo...","tank top, black, jersey, short, scoop neck, lo...","tank top, black, jersey, short, scoop neck, lo...","tank top, black, jersey, short sleeves, scoop ..."
4,00725_00.jpg,"Garment category: T-shirt, Colors: light beige...","T-shirt, light beige, not specified, 3/4, crew...","T-shirt, light beige, 3/4, crew neck, slim, so...","T-shirt, light beige, 3/4 sleeves, crew neck, ..."


In [37]:
print(f"Training Set Shape: {train_df.shape}")
print(f"Validation Set Shape: {val_df.shape}")
print(f"Testing Set Shape: {test_df.shape}")

Training Set Shape: (10482, 5)
Validation Set Shape: (1165, 5)
Testing Set Shape: (2032, 5)


In [38]:
train_json = (
    train_df[["file_name", "caption_v3"]].rename(columns={"caption_v3": "raw_caption"})
)

train_json.to_json(
    "train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [39]:
val_json = (
    val_df[["file_name", "caption_v3"]].rename(columns={"caption_v3": "raw_caption"})
)

val_json.to_json(
    "val.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [40]:
test_json = (
    test_df[["file_name", "caption_v3"]].rename(columns={"caption_v3": "raw_caption"})
)

test_json.to_json(
    "test.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [41]:
with open("train.jsonl", "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline())

{"file_name":"08945_00.jpg","raw_caption":"camisole, black, lace and possibly silk, strapless, v-neck with lace trim, form-fitting, solid, lace trim, text on the garment"}

{"file_name":"06064_00.jpg","raw_caption":"blouse, red, crepe-like, short sleeves, scoop neck, loose, solid, ONLY, zipper closure at back, casual"}

{"file_name":"02638_00.jpg","raw_caption":"T-shirt, gray, cotton, short sleeves, scoop neckline, loose fit, solid color, casual style"}

